
# 01 — Dataset Exploration

## Objective

The objective of this notebook is to explore and understand the
intracranial EEG (iEEG) dataset before performing signal preprocessing
or statistical analysis.

The exploration includes:

- Understanding the dataset structure.
- Inspecting participants and recordings.
- Exploring recording metadata.
- Understanding channel information.
- Exploring stimulation events.

## Dataset

- Dataset ID: ds006519
- Modality: Intracranial EEG (iEEG)
- Format: BIDS
- Source: https://openneuro.org/datasets/ds006519

## Important note

This notebook is exploratory. No signal filtering, artefact removal,
or scientific conclusions will be performed at this stage.



## 1. Import libraries

We use Python libraries to inspect the dataset, manipulate metadata,
and visualise the recordings.

- pathlib: navigate directories and locate files.
- json: read JSON metadata files.
- pandas: work with tabular data.
- numpy: numerical operations.
- matplotlib: visualise data.
- mne: load and inspect electrophysiological recordings.


In [ ]:

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne


In [ ]:

# Display library versions

print(f"MNE version: {mne.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")


In [ ]:

# Define the project and dataset paths

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"

# Check that the dataset directory exists

assert RAW_DATA_DIR.exists(), "Raw dataset directory was not found."

print(f"Project directory: {PROJECT_ROOT}")
print(f"Raw dataset directory: {RAW_DATA_DIR}")



## 2. Dataset overview

Before selecting an individual recording, we first investigate
the overall dataset composition.

We will identify:

- The number of participants.
- The number of iEEG recordings.
- The available recording formats.
- The distribution of recordings across participants.

This helps understand the dataset and select a representative
recording for further exploration.


In [ ]:

participants_file = RAW_DATA_DIR / "participants.tsv"

participants = pd.read_csv(participants_file, sep="\t")

print(f"Number of participants: {len(participants)}")

display(participants)


As can be seen here, there were 23 participants. But two of them were excluded for various reasons.

In [ ]:

# Find all iEEG directories

ieeg_directories = sorted(RAW_DATA_DIR.glob("sub-*/ses-*/ieeg"))


print(f"Participants with iEEG directories: {len(ieeg_directories)}")

for directory in ieeg_directories:
    print(directory)


In [ ]:

# Identify common electrophysiological recording formats

signal_extensions = [".edf", ".vhdr", ".set", ".fif"]


recording_files = sorted(
    file
    for directory in ieeg_directories
    for file in directory.iterdir()
    if file.suffix.lower() in signal_extensions
)

print(f"Number of recording files: {len(recording_files)}")

for file in recording_files[:20]:
    print(file)


The las two cells shows that there are 35 ieeg sessions, meaning that some participants have more than one session.
Also, it is shown that there are 35 signal recordings, which means that every session had a recording. 

## 2. Recording metadata

The recording metadata describes the technical properties of the
iEEG acquisition, including sampling frequency, measurement units,
and recording reference.

Understanding these properties is essential before preprocessing
or analysing the signal.

In [ ]:
ieeg_metadata_path = (
    RAW_DATA_DIR
    / "sub-01"
    / "ses-01"
    / "ieeg"
    / "sub-01_ses-01_task-dcs_ieeg.json"
)

with open(ieeg_metadata_path, "r") as f:
    ieeg_metadata = json.load(f)

ieeg_metadata

In [ ]:
print("Sampling frequency:", ieeg_metadata.get("SamplingFrequency"))
print("Power line frequency:", ieeg_metadata.get("PowerLineFrequency"))
print("Recording reference:", ieeg_metadata.get("iEEGReference"))

## 3. Channel metadata

The channels.tsv file describes the individual
recorded channels.

We inspect channel names, types, units and status
to understand which signals are available and
whether any channels have been marked as problematic.

In [ ]:
channels_path = (
    RAW_DATA_DIR
    / "sub-01"
    / "ses-01"
    / "ieeg"
    / "sub-01_ses-01_task-dcs_channels.tsv"
)

channels = pd.read_csv(channels_path, sep="\t")

print("Number of channels:", len(channels))
print("\nAvailable columns:")
print(channels.columns.tolist())

display(channels)

In [ ]:
print("Channel types:")
print(channels["type"].value_counts(dropna=False))

print("\nMeasurement units:")
print(channels["units"].value_counts(dropna=False))

In [ ]:
print(
    channels["sampling_frequency"]
    .value_counts(dropna=False)
)

In [ ]:
print("Channel status:")
print(channels["status"].value_counts(dropna=False))

print("\nStatus descriptions:")
print(channels["status_description"].value_counts(dropna=False))

In [ ]:
for column in [
    "reference",
    "low_cutoff",
    "high_cutoff",
    "notch"
]:
    print(f"\n{column}:")
    print(channels[column].value_counts(dropna=False))

On the cells above, it is shown that there are 31 channels.
All of the channels are SEEG, have the same sampling frequency of 4096 (as seen on the matadata of the session). The status of all the channels is marked as "good" and there is not a low or high cut-off or a notch filter are documented applied.
However, this does not necessarily mean that no hardware or acquisition-level filtering was applied.

## 4. Event metadata

The events.tsv file describes experimental events occurring
during the recording.

We inspect the event types, onset times and durations to
understand the stimulation protocol and identify relevant
time points for subsequent signal analysis.

In [ ]:
events_path = (
    RAW_DATA_DIR
    / "sub-01"
    / "ses-01"
    / "ieeg"
    / "sub-01_ses-01_task-dcs_events.tsv"
)

events = pd.read_csv(events_path, sep="\t")

print("Number of events:", len(events))
print("\nColumns:")
print(events.columns.tolist())

display(events)

The events table identifies two 5-second epochs within each continuous iEEG recording:

- **Pre-stimulation (`prestim`):** Baseline period used to characterise neural activity before electrical stimulation.
- **Post-stimulation (`poststim`):** Period used to investigate neural activity following stimulation.

According to the dataset README, each recording contains approximately 24 seconds of iEEG data: 9–10 seconds before stimulation, up to 5 seconds of stimulation, and 9–10 seconds after stimulation.

The event metadata also documents stimulation parameters, including stimulation contacts, frequency, intensity and clinical effect.

**Purpose:** Identify the baseline and post-stimulation periods that will be used for subsequent signal analysis and connectivity comparisons.

In [ ]:
electrodes_path = (
    RAW_DATA_DIR
    / "sub-01"
    / "ses-01"
    / "ieeg"
    / "sub-01_ses-01_electrodes.tsv"
)

electrodes = pd.read_csv(electrodes_path, sep="\t")

display(electrodes.head())

Electrode metadata — electrodes.tsv
- Contains spatial coordinates (x, y, z) and metadata for individual electrode contacts.
- Each contact has an identifier (name) and belongs to an electrode group (group).
- Includes hemisphere and anatomical/clinical labels: white matter (WM), seizure-onset zone (SOZ) and epileptogenic zone (EZ).
- Coordinates describe electrode locations, not signal amplitudes.
- Coordinate interpretation requires checking coordsystem.json

In [ ]:
coordsystem_path = (
    RAW_DATA_DIR
    / "sub-01"
    / "ses-01"
    / "ieeg"
    / "sub-01_ses-01_coordsystem.json"
)

with open(coordsystem_path, "r") as f:
    coordsystem = json.load(f)

coordsystem

## Conclusion

This notebook explored the structure and metadata of the ds006519 intracranial EEG dataset.

The exploration covered:

- Dataset organisation into participants, sessions and recordings.
- BrainVision file structure and BIDS metadata.
- Recording properties, including sampling frequency, channel information and acquisition parameters.
- Stimulation-related event annotations, identifying pre-stimulation and post-stimulation epochs.
- Electrode metadata, including contact identifiers, anatomical classifications and spatial coordinates.

The dataset contains SEEG recordings acquired during direct cortical electrical stimulation. Its metadata provides the information needed to identify recording channels, stimulation conditions, relevant time periods and electrode locations.

**Next step:** Load and inspect the actual SEEG signals using MNE-Python, beginning with a representative recording.